In [7]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while not (project_root / "Application" / "PF_wrapper.py").is_file() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "Application"))
import numpy as np
import PF_wrapper as PF

In [8]:
DEFAULT_SEED = 42
dir1 = "training_output"
dir2 = "training_predictions"

In [9]:
PF.train(
    train_file="outlier_benchmarks/AcademicAnomaly/train.npy",
    test_file="outlier_benchmarks/AcademicAnomaly/test.npy",
    reader_type="NPY",
    seed=DEFAULT_SEED, # this is not fully deterministic (yet)
    forest_mode="isolation",
    #train_labels="outlier_benchmarks/AcademicAnomaly/train_labels.npy",  # we *could* use labels, but this problem is unsupervised.
    #test_labels="outlier_benchmarks/AcademicAnomaly/test_labels.npy",
    numeric_data=True,
    #subsample_dimensions=True,
    #dimension_selection_strategy="sqrt",
    distances=["euclidean"],
    num_trees=100,
    num_workers=-1,  # use -1 for all processors.
    #return_enhanced_outputs=True,
    return_ood_scores=True,
    return_proximities=False,
    return_predictions=True,
    max_depth=11, #log_2(N)
    #numeric_storage="float",
    use_vector_api=True,  # for euclidean distance and standardization, this is appropriate (faster).
    early_abandon_distances=False,  # probably better for ood scoring?? Training is faster with early abandoning.
    save_model=True, # This is only needed if model persistence is required (used by PF.predict).
    model_name="MyModel",
    standardization="Z_SCORE",
    standardization_scope="PER_DIMENSION",
    output_directory=dir1,
    purity="ISOLATION_MIN_CHILD",
    # bootstrap_trees=False, # this effectively disables GAP proximities. Training is faster with bootstrapping.
    #repeats=10,
)

0

In [10]:
PF.predict(
    dir1 + "/MyModel",
    "outlier_benchmarks/AcademicAnomaly/test.npy",
    reader_type="NPY",
    numeric_data=True,
    num_workers=4,
    #return_enhanced_outputs=True,
    return_ood_scores=True,
    return_predictions=True,
    use_vector_api=True,
    standardization="Z_SCORE",
    standardization_scope="PER_DIMENSION",
    output_directory=dir2,
)

0

In [11]:
# we'll probably want to compare isolation and ood scores with the labels.
# (potentially interesting to scatter plot ood and isolation scores, color by "true labels"
# the npy files can be loaded directly into python via numpy.
X_train = np.load("outlier_benchmarks/AcademicAnomaly/train.npy")

In [12]:
print("dtype:", X_train.dtype)
print("shape:", X_train.shape)
print("means:", np.mean(X_train, axis=0))
print("stds:", np.std(X_train, axis=0))
print("mins:", np.min(X_train, axis=0))
print("maxs:", np.max(X_train, axis=0))

dtype: float64
shape: (2327, 24)
means: [1.56043231e+00 6.21955853e+02 2.62689931e+02 8.41139407e+00
 3.25158445e+02 1.53673847e+00 1.59107800e+01 1.76136574e+00
 9.29543045e+02 1.54875243e+02 1.71715621e+00 8.94814592e+02
 7.87734623e+00 1.44080971e+02 2.63837318e+02 1.22062861e+01
 1.72728748e+00 1.28403643e+00 7.35382751e-01 3.07810985e-01
 2.50048236e+02 1.69647499e+00 1.88912165e+02 1.36468128e-04]
stds: [1.23999691e+00 1.21560133e+02 4.75258321e-01 3.60792728e-02
 5.63306539e+00 1.17574270e+00 7.15305924e+00 8.89593813e-01
 7.25219107e+01 8.70086019e-01 3.09495026e-03 6.97501061e+01
 1.93961473e-03 8.92025865e-01 1.42025906e+00 1.43063769e-01
 2.84301563e-03 1.39159430e-02 5.78352622e-04 9.00556493e-04
 6.22930844e-01 5.32062063e-02 5.42321610e-01 3.70684587e-05]
mins: [0.000000e+00 4.881866e+02 2.604460e+02 8.351256e+00 3.151243e+02
 0.000000e+00 7.490876e-01 0.000000e+00 7.883796e+02 1.526275e+02
 1.703905e+00 7.779543e+02 7.871251e+00 1.419636e+02 2.610228e+02
 1.172776e+01 1.